# Machine Learning Pipeline

Now that we have experience preparing data for input to machine learning libraries, the next step will be to train, tune, and test a model.  You will perform all three of these steps in this hands-on activity.

The assignment consists of the following steps:

1. Load two datasets and prepare their representations and labels for model input. 
2. Split the data into training and testing.
3. Select a model, and identify the parameters to tune.
4. Tune the model.
5. Evaluate the model's performance.

In [23]:
import logging
logging.getLogger("scapy.runtime").setLevel(logging.ERROR)

from netml.pparser.parser import PCAP
from netml.utils.tool import dump_data, load_data

import pandas as pd
import numpy as np


## Convert the Packet Capture Into Flows

1. Load the two packet captures for HTTP requests and Log4j scan, 
2. convert them into traffic flows, 
3. generate features from the flow,  
4. label the traffic,
5. normalize your labeled features into a 2D matrix

In [24]:
# 1 Loading two packet captures
import pandas as pd
ldf = pd.read_csv("data/log4j.csv.gz")
hdf = pd.read_csv("data/http.csv.gz")
ldf

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,2021-12-15 15:35:00.237882,zl-lax-us-gp3-wk106b.internet-census.org,91.247.71.198.host.secureserver.net,TCP,74,57468 > 80 [SYN] Seq=0 Win=29200 Len=0 MSS=146...
1,2,2021-12-15 15:35:00.237939,91.247.71.198.host.secureserver.net,zl-lax-us-gp3-wk106b.internet-census.org,TCP,74,"80 > 57468 [SYN, ACK] Seq=0 Ack=1 Win=65160 Le..."
2,3,2021-12-15 15:35:00.249425,zl-lax-us-gp3-wk106b.internet-census.org,91.247.71.198.host.secureserver.net,TCP,66,57468 > 80 [ACK] Seq=1 Ack=1 Win=29312 Len=0 T...
3,4,2021-12-15 15:35:00.249475,zl-lax-us-gp3-wk106b.internet-census.org,91.247.71.198.host.secureserver.net,HTTP,271,GET / HTTP/1.1
4,5,2021-12-15 15:35:00.249525,91.247.71.198.host.secureserver.net,zl-lax-us-gp3-wk106b.internet-census.org,TCP,66,80 > 57468 [ACK] Seq=1 Ack=206 Win=65024 Len=0...
...,...,...,...,...,...,...,...
80838,80839,2021-12-20 14:41:44.651279,172.70.178.251,91.247.71.198.host.secureserver.net,TCP,54,"35730 > 80 [FIN, ACK] Seq=503 Ack=336 Win=6758..."
80839,80840,2021-12-20 14:41:44.651328,91.247.71.198.host.secureserver.net,172.70.178.251,TCP,54,80 > 35730 [ACK] Seq=336 Ack=504 Win=64128 Len=0
80840,80841,2021-12-20 14:41:44.820637,91.247.71.198.host.secureserver.net,172.70.126.39,TCP,54,"80 > 62584 [FIN, ACK] Seq=335 Ack=586 Win=6412..."
80841,80842,2021-12-20 14:41:44.881694,172.70.126.39,91.247.71.198.host.secureserver.net,TCP,54,"62584 > 80 [FIN, ACK] Seq=586 Ack=336 Win=6758..."


In [25]:
# 2 convert into traffic flows
# Everything here needs packet-level data: one row per (Source, Destination)
# pair, with the raw aggregations only. No derived quantities.
def to_flows(df):
    df = df.copy()
    df['Time'] = pd.to_datetime(df['Time'])
 
    flows = df.groupby(['Source', 'Destination']).agg(
        packets=('Length', 'count'),
        bytes=('Length', 'sum'),
        avg_bytes_per_packet=('Length', 'mean'),
        packet_length_std=('Length', 'std'),
        min_packet_length=('Length', 'min'),
        max_packet_length=('Length', 'max'),
        n_protocols=('Protocol', 'nunique'),
        main_protocol=('Protocol', lambda s: s.mode().iat[0]),
        start=('Time', 'min'),
        end=('Time', 'max'),
    ).reset_index()
 
    # std of a single packet is undefined
    flows['packet_length_std'] = flows['packet_length_std'].fillna(0)
    return flows
 

In [26]:
# 3 Generate features from the flow
# Operates only on a flow table. Nothing here touches the packet log, so it
# can be re-run, extended, or tested without re-reading the capture.
 
def add_features(flows):
    f = flows.copy()
 
    f['duration'] = (f['end'] - f['start']).dt.total_seconds()
 
    # single-packet flows have duration 0; NA keeps the division finite
    d = f['duration'].replace(0, pd.NA)
    f['packets_per_second'] = f['packets'] / d
    f['bytes_per_second'] = f['bytes'] / d
 
    f['percent_of_total_bytes'] = f['bytes'] / f['bytes'].sum() * 100
    f['percent_of_total_packets'] = f['packets'] / f['packets'].sum() * 100
    f['packet_length_range'] = f['max_packet_length'] - f['min_packet_length']
    f['length_cv'] = f['packet_length_std'] / f['avg_bytes_per_packet']
 
    return f
 
 
def add_host_features(f):
    """Per-source behaviour — needs the whole flow table, not one row."""
    f = f.copy()
    by_src = f.groupby('Source')
    f['src_n_destinations'] = f['Source'].map(by_src['Destination'].nunique())
    f['src_n_flows'] = f['Source'].map(by_src.size())
    f['src_total_bytes'] = f['Source'].map(by_src['bytes'].sum())
    return f
 
 
def build(flows):
    return add_host_features(add_features(flows))

In [27]:
ldf_features = build(to_flows(ldf))
hdf_features = build(to_flows(hdf))

In [28]:
# Label the traffic
ldf_features['label'] = 1   # log4j scan
hdf_features['label'] = 0   # benign http

df = pd.concat([ldf_features, hdf_features], ignore_index=True)

# Columns that encode which capture a flow came from, not what it did
LEAKY = ['duration', 'packets_per_second', 'bytes_per_second',
         'percent_of_total_bytes', 'percent_of_total_packets',
         'src_n_destinations', 'src_n_flows', 'src_total_bytes',
         'start', 'end', 'Source', 'Destination']

X = df.drop(columns=LEAKY + ['label'])
X = pd.get_dummies(X, columns=['main_protocol'], drop_first=True)
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
y = df['label']

print(X.shape, y.value_counts().to_dict())

(9416, 47) {1: 9219, 0: 197}


In [30]:
# Normalize the labeled features into a 2D matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# tame the heavy right skew before scaling
for c in ['packets', 'bytes']:
    X[c] = np.log1p(X[c])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train
X_test_scaled  = scaler.transform(X_test)        # apply to test

print(X_train_scaled.shape, X_test_scaled.shape)

(7532, 47) (1884, 47)


## Evaluating a Machine Learning Model

The goal of supervised learning is to train a model that takes examples and predicts labels for these examples that are as close as possible to the actual labels. For instance, in this example above, a model might take features from a traffic trace and predict whether the traffic constitutes regular web traffic or a scan.

How do you measure whether the model is succeeding if you don't know the true labels for new observations? The way to solve this problem is to test the performance of the trained algorithm on additional data that it has never seen, but for which you already know the correct labels. 

This requires that you train the algorithm using only a portion of the entire labeled dataset (the **training set**) and withold the rest of the labeled data (the **test set**) for testing how well the model generalizes to new information. 

To evaluate the model, we will need to split the data into train and test sets.

### Split into Training and Test Sets

Split your data into a training and test set using scikit-learn. A common split is to train on 80% of your data, while withholding 20% of the data. 

In [31]:
# Did it in the normalize labels
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True).round(4).to_dict())
print(y_test.value_counts(normalize=True).round(4).to_dict())

(7532, 47) (1884, 47)
{1: 0.979, 0: 0.021}
{1: 0.9793, 0: 0.0207}


### Training Your Model

Now that you have split your data into training and testing sets, you are ready to train and evaluate a model. 

Import a machine learning model of your choice, use your training set to train the model, and use the test set to evaluate it. 

### Test Your Trained Model

You can now evaluate how well your trained model works.  There are several valuable ways to visualize your results. You might use techniques such as a confusion matrix, or a receiver operating characteristic (ROC) curve. Below we will gain some experience plotting both of those.  This [documentation](https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_display_object_visualization.html) may help you with plotting these results.

#### Confusion Matrix 

A confusion matrix is a one way to understand errors of different types. We can see a lot of examples off diagonal, suggesting a fair number of incorrect answers.

#### Receiver Operating Characteristic

Some models can output different classes based on a threshold that is set for the decision. 

#### Area Under the Curve (AUC)

From the ROC above, you can also compute a metric called the area under the curve (AUC). Visually, this is the area under the curve that you just plotted. You could see, intuitively, that the "best" performance should yield an AUC of 1, and the worst performance would yield an AUC closer to 0.5.

Scikit learn also has a function for computing AUC.  Compute the area under the curve.

## Thought Question

Which evaluation model is more appropriate, and when (i.e., under what circumstances)? When might you care more about looking at the confusion matrix (or model accuracy) vs. the ROC, or the area under the curve?